In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

feats = pd.read_csv("data/processed/features.csv")
FULL = [c for c in feats.columns if c not in ("id", "label", "source", "split")]
MINIMAL = ["n_vertices_4", "circle_fill", "rect_fill", "rect_aspect", "hu1"]
SOURCES = ["generated", "kaggle_2d", "kaggle_simple"]
tr = feats[feats.split == "train"]

models = {
    "LogReg": make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=2000, class_weight="balanced")),
    "DecisionTree": DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=0),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
    "SVM": make_pipeline(StandardScaler(), SVC(C=10, kernel="rbf", class_weight="balanced")),
    "RandomForest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=0),
}

# 1) model x held-out source (minimal features)
rows_detail = []
for name, model in models.items():
    for src in SOURCES:
        a, b = tr[tr.source != src], tr[tr.source == src]
        p = clone(model).fit(a[MINIMAL], a.label).predict(b[MINIMAL])
        rows_detail.append({"model": name, "held_out": src,
                            "accuracy": accuracy_score(b.label, p),
                            "f1_present": f1_score(b.label, p, labels=sorted(b.label.unique()),
                                                   average="macro")})
detail = pd.DataFrame(rows_detail)

# 2) single-feature transfer across sources
rows_feat = []
for f in FULL:
    r = {"feature": f}
    for src in SOURCES:
        a, b = tr[tr.source != src], tr[tr.source == src]
        m = DecisionTreeClassifier(max_depth=4, random_state=0).fit(a[[f]], a.label)
        r[src] = accuracy_score(b.label, m.predict(b[[f]]))
    rows_feat.append(r)
transfer = pd.DataFrame(rows_feat).set_index("feature")

os.makedirs("outputs", exist_ok=True)
detail.to_csv("outputs/by_source_detail.csv", index=False)
transfer.to_csv("outputs/feature_transfer.csv")
print(detail.pivot(index="model", columns="held_out", values="accuracy").round(3))
print(transfer.round(3))

held_out      generated  kaggle_2d  kaggle_simple
model                                            
DecisionTree      1.000        1.0          0.996
KNN               1.000        1.0          0.951
LogReg            0.999        1.0          0.997
RandomForest      1.000        1.0          1.000
SVM               1.000        1.0          0.967
              generated  kaggle_2d  kaggle_simple
feature                                          
area              0.742      0.791          0.758
perimeter         0.544      0.596          0.288
circularity       0.922      0.834          0.943
solidity          0.587      0.736          0.561
circle_fill       0.999      0.999          0.994
rect_fill         0.999      0.995          0.976
rect_aspect       0.660      0.614          0.283
n_vertices_2      0.955      0.981          0.935
n_vertices_4      1.000      1.000          0.821
hu1               1.000      1.000          0.999
hu2               0.375      0.400          0.128
